<a href="https://colab.research.google.com/github/ifusppos/ingresso_pgifusp/blob/main/EUF_Melhor-Nota.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Definição do Ambiente

In [ ]:
%%capture
!pip install fuzzywuzzy
!pip install python-Levenshtein
!pip install recordlinkage
!pip install unidecode
!pip install joypy

In [ ]:
# Importa as bibliotecas Google Sheets
import gspread
import pandas as pd
import recordlinkage
import re
import unidecode
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import joypy
from google.colab import auth
auth.authenticate_user()
from google.auth import default
from gspread_dataframe import set_with_dataframe
creds, _ = default()
gc = gspread.authorize(creds)
# teste de commit perfil ifpos 2

In [ ]:
# VARIÁVEIS

# Dados de onde está localizada a planilha com as Notas do EUF (ORIGINAL: '1G2r-O-adeP_YjRzlR1U0IskSSWv0OHy5')
id_pasta_1 = '16KwgyZ2KySE1V93mQsdy5wRATwvZuVy3'  # id da pasta onde está localizada a Planilha no seu Google drive
nome_planilha_1 = 'Notas EUF - 2006 em diante' # Nome da Planilha dentro do seu Google drive


# Dados de onde está localizada a planilha com as inscrições do CPGpar (ORIGINAL: '1G2r-O-adeP_YjRzlR1U0IskSSWv0OHy5')
id_pasta_2 = '16KwgyZ2KySE1V93mQsdy5wRATwvZuVy3'  # id da pasta onde está localizada a Planilha no seu Google drive
nome_planilha_2 = 'Inscrições_CPGpar' # Nome do Planilha dentro do seu Google drive


# Nome da edição atual que será usado para salvar os arquivos dos aprovados e reprovados
edicao = "1_2025"  # Nome do Planilha dentro do seu Google drive

In [ ]:
# Função para unir nomes semelhantes, com base em uma %, entre o dfA (CPGpar) com o dfB(Notas do EUF)
# Retorna dfA combinado com as colunas escolhidas (colunas_B) de dfB.
def merge_RL(dfA,dfB,nome_B,colunas_B,nivel):
  # candidatos
  indexer = recordlinkage.Index()
  indexer.full()
  canditados = indexer.index(dfA, dfB)
  # Comparação
  compare_cl = recordlinkage.Compare()
  compare_cl.string('Nome_', 'Nome', method='levenshtein', threshold=nivel, label= 'conta' )
  features = compare_cl.compute(canditados, dfA, dfB)
  matches = features[features.sum(axis=1) >= 1]
  matches = matches.reset_index()
  dfA = dfA.reset_index()

  merge1=dfA.merge(matches,
                left_on='index',
                right_on='level_0',
                how='left')

  matches=merge1.merge(dfB[colunas_B],
                left_on = 'level_1',
                right_index=True,
                suffixes=(None, None),
                how='left'
  )
  matches = matches.drop(columns = ['index','level_0','level_1','conta'])
  return matches

# Funçao que recebe um dataframe e sua coluna com as notas
# Retorna o mesmo dataframe com uma nova coluna (_Norm) com a nota normalizada
def notaNormalizada(df,coluna):
  df = eval(df)
  notas = pd.to_numeric(df[coluna])
  notas = df[coluna].astype(float)
  df[coluna+'_Norm'] = notas*5/notas.mean()
  return df

# Funçao que recebe uma  string e um caracter (subtring)
# Retorna a mesma string sem o caracter
def limpa_nota(string,subtring):
  # transforma
  string = str(string).replace(subtring,"")
  return string

# Funçao que recebe a nota normalizada
# Retorna "Aprovado"se >4 ou "Sem nota suficiente"
def verificaNotaNorm(nota_normalizada):
  if nota_normalizada >=4.00:
    return "Aprovado"
  else:
    return "Sem nota suficiente"

# Funçao que recebe a nota normalizada
# Retorna "Aprovado"se >6 ou "Sem nota suficiente"
def verificaNotaNormDD(nota_normalizada):
  if nota_normalizada >=6.00:
    return "Aprovado"
  else:
    return "Sem nota suficiente"

# Funçao que recebe o texto
# Retorna o texto sem ","
def removeVirgulaGS(rows):
  for i, x in enumerate(rows):
      for j, a in enumerate(x):
          if ',' in a:
              rows[i][j] = a.replace(',', '.')
  return rows

# Funçao que recebe um dataframe
# Retorna o df
def etiquetaDFinteiro(df,etiqueta):
  etiqueta=str("_"+etiqueta+"")
  df.columns=[x + etiqueta for x in list(df.columns)]
  return df


# Funçao que recebe o dataframe e 2 colunas
# Retorna uma lista com os valores do dataframe
def subtringLista(lista,substring1, substring2=None):
  lista_filtrada = []
  for string in lista:
    if substring2 != None:
      if substring1 in string and substring2 not in string:
        lista_filtrada.append(string)
      else:
        pass
    else:
      if substring1 in string:
        lista_filtrada.append(string)
      else:
        pass
  return lista_filtrada


# Funçao que recebe um texto
# Retorna o texto sem aspas
def extraiNomePlanilha(s):
  padrao = "\'(.*?)\'"
  return re.search(padrao,s).group(1)


# Funçao que recebe um texto
# Retorna o texto com a primeira letra de cada palavra maiúscula
def capitalizarNome(string):
  return ' '.join([s.capitalize() for s in string.split()])


# Funçao que recebe um dataframe e colunas especificas
# retorna o mesmo dataframe com o sufixo "_" no nome das colunas específicas
def etiqueta_df(df, colunas, etiqueta):
  etiqueta=str("_" + etiqueta)
  colunas_etiq = [x + etiqueta for x in colunas]
  dicionario = dict(zip(colunas, colunas_etiq))
  df=df.rename(columns=dicionario)
  return df


# Funçao que recebe um dataframe
# Retorna o arquivo em google sheets
def DFParaGsheet(nome_arquivo, nome_planilha, df):
  n_col, n_linha = df.shape
  lista_arquivos = [d['name'] for d in gc.list_spreadsheet_files()]
  if nome_arquivo not in lista_arquivos:
    sh=gc.create(nome_arquivo,folder_id=id_pasta_1)
  else:
    sh = gc.open(nome_arquivo)
  planilha = sh.add_worksheet(title=nome_planilha, rows=n_linha, cols=n_col)
  planilha.update([df.columns.values.tolist()] + df.values.tolist())


# Funçao que recebe um texto
# Retorna Mestrado, Doutorado ou Dou.Direto
def corrigeCurso(string):
  if string =='inscricaomd':
    return "Mestrado"
  elif string == 'incricaodr':
    return "Doutorado"
  else:
    return "Dou.Direto"

# Funçao que recebe um texto
# Retorna Sim ou Não
def corrigeConcorreBolsa(string):
  if string == 'Desejo concorrer à Bolsa':
    return "Sim"
  else:
    return "Não"


# Funçao que recebe um texto
# Retorna o texto sem espaços adicionais
def removeMultEspacos(string: str) -> str:
    string_arrumada = re.sub(' +', ' ', string)
    return string_arrumada


# Funçao que recebe um texto
# Retorna o texto sem acentos
def removeAcentos(string: str) -> str:
    string_arrumada = unidecode.unidecode(string)
    return string_arrumada


# Funçao que recebe um texto
# Retorna o texto sem acentos
# Retorna o texto sem acentos e sem espaços a direita e a esquerda
def processa_texto(df: pd.DataFrame, coluna: list) -> pd.DataFrame:
    coluna=str(coluna)
    df[coluna] = df[coluna].apply(str.rstrip) # remove espaços a direita em strings
    df[coluna] = df[coluna].apply(str.lower) # strings em letas minúsculas
    df[coluna] = df[coluna].apply(str.lstrip) # remove espaços a esqueda em strings
    df[coluna] = df[coluna].apply(removeAcentos) # remove acentuação
    df[coluna] = df[coluna].apply(removeMultEspacos) # remove espaços multiplos: maior ou igual a 2
    return df

# Funçao que recebe um texto
# Retorna o texto tudo em minúsculo, sem acentos, sem espaços a direita e a esquerda e sem multiplo espaços
def processaString(string) -> str:
    string = string.lower()
    string = removeMultEspacos(string)# remove espaços multiplos: maior ou igual a 2
    string = string.strip() # remove espaços a direita  e esqueda em strings
    string = removeAcentos(string)
    return string

# Importação dos dados

In [ ]:
# Abre planilha do Google Sheet
sh1 = gc.open(nome_planilha_1, folder_id = id_pasta_1)


# Lista de listas que armazena todos os dados de cada aba em dataframes separados
dic={}

lista_p = [extraiNomePlanilha(str(i)) for i in sh1.worksheets()] # Extrai o nome de "todas as abas" dentro da planilha e salva na lista_p
for p in lista_p:
  planilha = sh1.worksheet(p)
  rows = planilha.get_all_values() # pega os valores preenchidas com algum valor das linhas da planilha
  rows = removeVirgulaGS(rows)
  dataframe=pd.DataFrame.from_records(rows[1:],columns=rows[0])
  dic[p] = dataframe  # armazena cada aba em uma lista

dfs_euf = list(dic.keys()) #  dfs_euf é o nome com todos os dataframes criados no dicionario "dic"
try:
  dfs_euf.remove('2021_2') #  remove edição cancelada do EUF caso ela esteja erroneamente na planilha de notas.
except:
  pass

In [ ]:
# Pega os documentos dos participantes do EUF nas planilhas de inscrição
for key in dfs_euf[:8]:
  planilha_inscritos = str('EUF-INSCRITOS_' + key.replace('_','.'))
  inscritos = gc.open(planilha_inscritos, folder_id = id_pasta_2)
  rows = inscritos.worksheet('Sheet1').get_all_values() # pega os valores preenchidas com algum valor das linhas da planilha
  rows = removeVirgulaGS(rows)
  dataframe=pd.DataFrame.from_records(rows[3:],columns=rows[2])
  dataframe = dataframe.rename(columns={'Número' : 'EUF CODE'})
  dic[key] = pd.merge(dic[key],dataframe[['EUF CODE','Documento']])
# dic['2024_1']

In [ ]:
# não precisa execultar
for key in dfs_euf[:8]:  #Verifica e lista todos as edições de EUF
  print(key)

In [ ]:
# não precisa execultar
dic['2025_1'] # apenas para visualizar os dados de alguma aba

In [ ]:
# Abre planilha do Google Sheet
sh2 = gc.open(nome_planilha_2, folder_id= id_pasta_2) # abre o arquivo Google Sheet
planilha2 = sh2.worksheet("Inscritos")

# ATENÇÃO!!! O método .get_all_values() pode falhar e exibir: exceeds grid limits
# renomeie a planilha adicionando underscore "_" em alguma parte no nome. Funciona, não se sabe porque.

rows = planilha2.get_all_values() # pega os valores preenchidas com algum valor das linhas da planilha
CPGPAR = pd.DataFrame.from_records(rows[1:], columns=rows[0]) # transforma em data frame do pandas


# Cabeçalho para trocar o nome de algumas colunas (dicionário)
novos_cabecalhos = {'curso no qual o candidato está se inscrevendo' : 'Curso',
                   'Instituição de onde vem' : 'Inst. origem',
                   'Selecione o EUF cuja nota você pretende utilizar para ingresso' : 'EUF indicado',
                   'Nota antiga do Exame EUF' : 'EUF antigo',
                   'Número de Inscrição' : 'Nº de inscrição',
                   'Declaro que' : "Concorre à bolsa"
}

# Aplica o cabeçalho
CPGPAR = CPGPAR.rename(columns=novos_cabecalhos)

CPGPAR['Curso'] = CPGPAR['Curso'].apply(corrigeCurso)
CPGPAR['Concorre à bolsa'] = CPGPAR['Concorre à bolsa'].apply(corrigeConcorreBolsa)
CPGPAR['Nome'] = CPGPAR['Nome'].apply(processaString)

In [ ]:
# não precisa rodar
CPGPAR # apenas para visualizar os dados de alguma aba

In [ ]:
dfA = CPGPAR
dfA = dfA.rename(columns={'Nome':'Nome_'})

for key in dfs_euf[0:8]: # OBSERVAÇÂO: Normalmente seria [:8], mas houve 2 edições extras do EUF
  print(key, )
  dic[key]['Nome'] = dic[key]['Nome'].apply(processaString)
  dic[key]['Percentil'] = dic[key]['Percentil'].str.replace('%','').astype(float)
  dic[key]['Percentil'] = dic[key]['Percentil'].astype(float)   # remover posteriormente
  dic[key]['Nota'] = dic[key]['Nota'].astype(float)

  # Calcula a média da edição
  media_edicao = dic[key]['Nota'].mean()
  print(key,media_edicao)
  dic[key]['Nota_norm'] = dic[key]['Nota'].apply(lambda x: x*5/media_edicao)

In [ ]:
  dic['2024_2']['Nota_norm'] = dic['2024_2']['Nota'].apply(lambda x: x*5/3.74) # Utilizar essa linha, essa edição teve leve diferença

In [ ]:
lista_colunas = ['Nome','Percentil','Nota','Nota_norm']# lista colunas de interesse
for key in dfs_euf[:8]:
  dfB = dic[key]
  # coluna_nome = key    # é necessário???
  # print(key)
  dfA = merge_RL(dfA,dfB,'Nome',lista_colunas,0.85) # 0.85 por padrão, depois rodar em menores.
  dfA = etiqueta_df(dfA, lista_colunas, key)


In [ ]:
dfA

# Tratamento dos Dados

In [ ]:
dfC = dfA.copy() # caso algo dê errado daqui para baixo, rode apenas esse celula

In [ ]:
# Ordena as colunas
cabecalhos = list(dfC.columns)

cabecalhos_p1=['Nome_',
                'Curso',
                'Nº de inscrição',
                'Inst. origem',
                'Email',
                'Concorre à bolsa',
                'EUF indicado',
                'EUF antigo',
                'Orientador Informado',
                'Tem carta de aceite?',
                'Orientador Credenciado?',
                'Estará credenciado na data de matrícula?',
                'Observações',
                'Data de Envio',
                'Sexo Biológico',
                'Identidade de Gênero',
                'Orientação Sexual',
                'Pessoa com Deficiência (PcD*)?',
                'Cor ou raça*',
                'Nível educacional da mãe ou genitor primário',
                'Onde estudou a mãe (ou genitor primário) no ensino médio?',
                'Onde estudou no ensino médio?']

cabecalhos_p2 = ['Nota_maxima',
                'Nota_norm_maxima',
                'Percentil_maximo',
                'euf_selecionado',
                'Situação'
                ]

cabecalhos_p3 = [cab for cab in cabecalhos if cab not in cabecalhos_p1 + cabecalhos_p2]
cabecalhos_p3.sort()
cabecalhos_ordenados = cabecalhos_p1 + cabecalhos_p2 + cabecalhos_p3
# cabecalhos_p3

In [ ]:
# não precisa rodar
cabecalhos_ordenados

In [ ]:
# Procura a maior nota
lista=list(dfC.columns)
string1 = 'Nota'
string2 = 'norm'
dfC['Nota_maxima'] = dfC[subtringLista(lista, string1, string2)].astype(float).max(axis=1)

string1 = 'Nota'
string2 = None
dfC['Nota_norm_maxima'] = dfC[subtringLista(lista, string1, string2)].astype(float).max(axis=1)

string1 = 'Percentil'
string2 = None
dfC['Percentil_maximo'] = dfC[subtringLista(lista, string1, string2)].astype(float).max(axis=1)

string1 = 'Percentil'

dfC['euf_selecionado'] = dfC[subtringLista(lista, string1, string2)].astype(float).idxmax(axis="columns")



# Verifica o curso e, se DD, aplica função alternativa (Not Norm Máx =>6)
dfC['Situação'] = dfC.apply(lambda linha: verificaNotaNorm(linha['Nota_norm_maxima']) if linha['Curso']!='Dou.Direto' else verificaNotaNormDD(linha['Nota_norm_maxima']) , axis=1)

dfC = dfC[cabecalhos_ordenados]

dfC = dfC.rename(columns = {'Nome_':'Nome',
                    'Pessoa com Deficiência (PcD*)?': 'Necessidades especiais',
                    'Nota_norm_maxima':'Nota norm. máx.',
                    'Cor ou raça*':'Cor ou raça',
                    'Nota_maxima': 'Nota máxima'
                    })



dfC['Nomes repetidos'] = dfC.groupby('Nome')['Nome'].transform('count')

nomes_duplicados = dfC[dfC['Nomes repetidos']>1]                        # Salva na variável a listas linhas repetidas

dfC = dfC.sort_values('Nº de inscrição').drop_duplicates(['Nome'], keep='last').reset_index()
dfC['Nome'] = dfC['Nome'].apply(capitalizarNome)

In [ ]:
dfC['Nomes repetidos'] = dfC.groupby('Nome')['Nome'].transform('count')
dfC['euf_selecionado'] = dfC['euf_selecionado'].apply(lambda x: str(x).replace('Percentil_',''))

In [ ]:
dfC[dfC['Nomes repetidos']>1]  # Verifica se existem nomes repetidos

In [ ]:
dfC = dfC.sort_values('Nº de inscrição').drop_duplicates(['Nome'], keep='last').reset_index().drop(columns=['index','level_0'])

In [ ]:
dfC.columns

In [ ]:
dfC['euf_selecionado'] = dfC['euf_selecionado'].replace({'nan' : 'EUF não encontrado'}) #EUF não encontrado
df_nao_encontrado = dfC[dfC['euf_selecionado'] == 'EUF não encontrado']
df_nao_encontrado = df_nao_encontrado[['Nome','Nota norm. máx.', 'euf_selecionado', 'Situação','Email']]
df_nao_encontrado.to_excel(f'INGRESSANTES_{edicao}_sem_EUF.xlsx',index=False)
df_nao_encontrado

In [ ]:
# df_reprov = dfC[['Nome','Nota norm. máx.','Percentil_maximo', 'euf_selecionado', 'Situação','Email']]
# OU
#df_reprov = dfC
df_reprov = dfC[dfC['Situação']!='Aprovado']
df_reprov = df_reprov[['Nome','Nota norm. máx.','Percentil_maximo', 'euf_selecionado', 'Situação','Email']]
df_reprov.to_excel(f'INGRESSANTES_{edicao}_insuficiente.xlsx',index=False)
df_reprov # Apenas printar

In [ ]:
# Gera o excel
df_aprov = dfC[['Nº de inscrição','Nome','Curso','Nota norm. máx.','Percentil_maximo','euf_selecionado','Concorre à bolsa', 'Situação','Email']]
df_aprov = df_aprov[df_aprov['Situação']=='Aprovado']
df_aprov.to_excel(f'INGRESSANTES - Aprovados - {edicao}.xlsx',index=False)

In [ ]:
df_euf_selecionado = df_aprov[df_aprov['Concorre à bolsa']=='Sim']

In [ ]:
# Gera o excel
df_euf_selecionado[['Nº de inscrição','Curso','euf_selecionado']].to_excel(f'INGRESSANTES - Lista de EUF selecionado - {edicao}.xlsx',index=False)
df_euf_selecionado

In [ ]:
# Inserir "Não respondeu" em celulas em branco do questionário socio econômico

colunas = ['Sexo Biológico',
           'Identidade de Gênero', 'Orientação Sexual', 'Necessidades especiais',
           'Cor ou raça', 'Nível educacional da mãe ou genitor primário',
           'Onde estudou a mãe (ou genitor primário) no ensino médio?',
           'Onde estudou no ensino médio?']
# dfC.columns
for coluna in colunas:
  dfC[coluna].replace('',"Não respondeu", inplace=True)

In [ ]:
# Gera o excel

dfC.to_excel(f'INGRESSANTES - EUFs e inscrições do CPGpar - {edicao}.xlsx', index=False)